# Read Parquet file

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("demo").getOrCreate()

In [4]:
ls

'1. Import Databricks and CSV Databricks.ipynb'
'1. Import Databricks and CSV.ipynb'*
'2. Json, Parquet Databricks.ipynb'
'2. Json, Parquet.ipynb'*
 test.csv
 user_detail.json*
 user_detail_multiline.json*
 user_detail_multiline_in_list.json*
 yellow_tripdata_2021-01.parquet*


In [9]:
file_location = "yellow_tripdata_2021-01.parquet"
file_type = "parquet"

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.parquet(file_location) 
df.printSchema()
df.show(10)

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

# Read Json File to DataFrame

Get data from https://github.com/indiacloudtv/pyspark_on_google_colab/tree/master

In [7]:
data = """
{"user_id":1,"user_name":"John","user_city":"London"}
{"user_id":2,"user_name":"Martin","user_city":"New York"}
{"user_id":3,"user_name":"Sam","user_city":"Sydney"}
{"user_id":4,"user_name":"Alan","user_city":"Mexico City"}
{"user_id":5,"user_name":"Jacob","user_city":"Florida"}
"""

file_location = "user_detail.json"
file_type = "json"


# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

df.show()

DataFrame[user_city: string, user_id: bigint, user_name: string]

+-----------+-------+---------+
|  user_city|user_id|user_name|
+-----------+-------+---------+
|     London|      1|     John|
|   New York|      2|   Martin|
|     Sydney|      3|      Sam|
|Mexico City|      4|     Alan|
|    Florida|      5|    Jacob|
+-----------+-------+---------+



But sometime json is in multiline

In [14]:
data = """
[
  {
    "user_id": 1,
    "user_name": "John",
    "user_city": "London"
  },
  {
    "user_id": 2,
    "user_name": "Martin",
    "user_city": "New York"
  },
  {
    "user_id": 3,
    "user_name": "Sam",
    "user_city": "Sydney"
  },
  {
    "user_id": 4,
    "user_name": "Alan",
    "user_city": "Mexico City"
  },
  {
    "user_id": 5,
    "user_name": "Jacob",
    "user_city": "Florida"
  }
]
"""

json_file_path_3 = "user_detail_multiline_in_list.json"
df = spark.read.json(path=json_file_path_3, multiLine=True)

df.show()
df.printSchema()

+-----------+-------+---------+
|  user_city|user_id|user_name|
+-----------+-------+---------+
|     London|      1|     John|
|   New York|      2|   Martin|
|     Sydney|      3|      Sam|
|Mexico City|      4|     Alan|
|    Florida|      5|    Jacob|
+-----------+-------+---------+

root
 |-- user_city: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_name: string (nullable = true)



We can also provide schema

In [19]:
from pyspark.sql.types import *

user_schema = StructType([
                     StructField("user_id", IntegerType(), True),
                     StructField("user_name", StringType(), True),
                     StructField("user_city", StringType(), True)
])

df = spark.read.json(path=json_file_path_3, multiLine=True, schema=user_schema)

df.printSchema()
df.show()

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- user_city: string (nullable = true)

+-------+---------+-----------+
|user_id|user_name|  user_city|
+-------+---------+-----------+
|      1|     John|     London|
|      2|   Martin|   New York|
|      3|      Sam|     Sydney|
|      4|     Alan|Mexico City|
|      5|    Jacob|    Florida|
+-------+---------+-----------+

